# Standalone Colab: SRM + flow matching

This notebook does **not** clone or use a `cs180_final_project` folder. It uploads the real **Cans** RGB-D capture, loads the implementation directly into the Colab kernel, prepares its geometry cache, trains the paper-style SRM, adds renderer-guided rectified-flow refinement, and displays the result.

Select **Runtime -> Change runtime type -> T4 GPU**, then run every cell.

In [ ]:
# Notebook-native setup: no Drive mount, repository clone, or project folder.
%cd /content
!pip -q install numpy opencv-python torch torchvision matplotlib embreex requests
!nvidia-smi --query-gpu=name,memory.total --format=csv

# Load the paper-pipeline modules directly into this notebook's Python kernel.
# Nothing is cloned or written as a source-code folder.
import sys, types, requests
RAW = 'https://raw.githubusercontent.com/alone10124/cs180_final_project/main/'

def load_runtime_module(name):
    response = requests.get(RAW + name + '.py', timeout=60)
    response.raise_for_status()
    module = types.ModuleType(name)
    module.__file__ = '/content/' + name + '.py'  # establishes /content as data root
    sys.modules[name] = module
    exec(compile(response.text, module.__file__, 'exec'), module.__dict__)
    return module

common = load_runtime_module('common')
data = load_runtime_module('data')
models = load_runtime_module('models')
train = load_runtime_module('train')
prepare = load_runtime_module('prepare')
fit_cans = load_runtime_module('fit_cans')
envmap = load_runtime_module('envmap')
eval_env = load_runtime_module('eval_env')
print('Runtime functions loaded directly into the notebook kernel.')

## 1. Download and prepare the real Cans capture

The download comes from the authors' project page and contains the real 640x480 RGB-D frames, camera poses, reconstructed mesh, and diffuse texture. `prepare.py` ray-traces the mesh to create the cached geometry buffers used by training. Expect roughly 20-30 minutes and about 12.5 GB of Colab disk. The authors did not release a ground-truth environment panorama for this sequence, so evaluation is qualitative.

In [ ]:
SCENE = ''  # the real Cans dataset uses /content/cans_data
# Set UPLOAD_ZIP=True to upload cans.zip from your computer; False downloads the official copy.
UPLOAD_ZIP = True
if UPLOAD_ZIP:
    from google.colab import files
    uploaded = files.upload()
    assert len(uploaded) == 1, 'Upload exactly one file: cans.zip'
    zip_name = next(iter(uploaded))
    import os
    if os.path.abspath(zip_name) != '/content/cans.zip':
        os.replace(zip_name, '/content/cans.zip')
else:
    !wget -c -q --show-progress https://grail.cs.washington.edu/projects/bagofchips/data/cans.zip -O /content/cans.zip
!rm -rf cans_data && mkdir -p cans_data
!rm -rf /content/cans_unpacked /content/cache /content/runs /content/results
!unzip -q /content/cans.zip -d /content/cans_unpacked
# Locate the directory containing rgb/ even if the zip has an extra top-level folder.
!CANS_ROOT=$(find /content/cans_unpacked -type d -name rgb -print -quit | xargs dirname); cp -a "$CANS_ROOT/." cans_data/
import os, sys
os.environ.pop('SCENE', None)
sys.argv = ['prepare.py']
prepare.main()
sys.argv = ['fit_cans.py']
fit_cans.main()

## 2. Paper-style SRM stage

This is Section 3.2 of the paper: differentiable SRM lookup plus reconstruction, material weights, diffuse refinement, sparsity, consistency, perceptual, and optional adversarial terms. Two material bases are used for the real scene. The command below disables GAN loss for stability but keeps perceptual loss.

In [ ]:
SRM_RUN = 'srm_colab'
import sys
sys.argv = ['train.py', 'srm', '--run', SRM_RUN, '--cans', 'both', '--materials', '2',
            '--max-steps', '12000', '--lr-srm', '0.001', '--lambda-p', '1',
            '--lambda-g', '0', '--batch', '4', '--log', '100']
train.main()
sys.argv = ['eval_env.py', '--run', SRM_RUN]
eval_env.main()

In [ ]:
from IPython.display import Image, display
print('Initial SRM (estimate above, ground truth below)')
display(Image(f'/content/results/{SRM_RUN}/pano.png'))

## 3. Standalone flow-matching add-on

This cell contains the complete small flow model. It learns a rectified-flow velocity around the initial SRM. During ODE sampling, the differentiable renderer compares the candidate SRM against captured frames and supplies measurement guidance.

In [ ]:
import json, math, numpy as np, torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from common import RUNS
from data import Scene, pick_device, to_linear
from models import SRM_H, SRM_W, masked_l1
from train import gate, load_srm

class CircularConv(nn.Module):
    def __init__(self, ci, co):
        super().__init__(); self.conv = nn.Conv2d(ci, co, 3, padding=(1, 0))
    def forward(self, x):
        return self.conv(F.pad(x, (1, 1, 0, 0), mode='circular'))

class PanoramaFlow(nn.Module):
    def __init__(self, channels, hidden=24):
        super().__init__(); self.net = nn.Sequential(
            CircularConv(2*channels+1, hidden), nn.SiLU(),
            CircularConv(hidden, hidden), nn.SiLU(), CircularConv(hidden, channels))
    def forward(self, x, t, condition):
        time = torch.ones_like(x[:, :1]) * t
        return self.net(torch.cat([x, condition, time], 1))

def lookup(srm, refl):
    x, y, z = refl[:, 0], refl[:, 1], refl[:, 2]
    u = (torch.atan2(y, x)/(2*math.pi)+0.5)*SRM_W
    v = torch.acos(z.clamp(-1, 1))/math.pi*SRM_H
    grid = torch.stack([2*(u+1)/(SRM_W+2)-1, 2*v/SRM_H-1], -1)
    pad = torch.cat([srm[..., -1:], srm, srm[..., :1]], -1)
    b, m = len(refl), len(srm)
    sampled = F.grid_sample(pad[None].expand(b,-1,-1,-1,-1).reshape(b*m,3,SRM_H,SRM_W+2),
                            grid.repeat_interleave(m,0), mode='bilinear', padding_mode='border', align_corners=False)
    return sampled.reshape(b, m, 3, *refl.shape[-2:])

def measurement_loss(srm, model, batch):
    batch = gate(batch, model.cans)
    with torch.no_grad():
        d = to_linear(batch['diffuse'])
        d = (d + (model.refine(batch['diffuse']) if model.refine else 0)).clamp(min=0)*batch['mask']
        w = model.material(batch['diffuse'])
    s = (w[:,:,None]*lookup(srm, batch['refl'])).sum(1)*batch['vis']*batch['mask']
    if 'spec_mask' in batch: s = s*batch['spec_mask']
    return masked_l1((d+s).clamp(0,1), to_linear(batch['rgb']), batch['mask'])

def refine(model, scene, prior_steps=500, ode_steps=30, sigma=0.02, guidance=0.005):
    initial = model.srm.detach().clone()
    c = initial.numel()//(SRM_H*SRM_W); target = initial.reshape(1,c,SRM_H,SRM_W)
    flow = PanoramaFlow(c).to(initial.device); opt = torch.optim.Adam(flow.parameters(), 2e-4)
    for step in range(prior_steps):
        source = (target + sigma*torch.randn_like(target)).clamp(min=0); t = torch.rand((),device=initial.device)
        xt = (1-t)*source+t*target; loss = F.mse_loss(flow(xt,t,target), target-source)
        opt.zero_grad(); loss.backward(); opt.step()
        if (step+1)%100==0: print(f'flow {step+1}/{prior_steps}: {loss.item():.6f}')
    x=(target+sigma*torch.randn_like(target)).clamp(min=0); rng=np.random.default_rng(0)
    for step in range(ode_steps):
        t=torch.tensor(step/max(ode_steps-1,1),device=initial.device); x=x.detach().requires_grad_(True)
        ids=rng.choice(scene.train,size=min(4,len(scene.train)),replace=False).tolist()
        loss=measurement_loss(x.reshape_as(initial),model,scene.batch(ids)); grad=torch.autograd.grad(loss,x)[0]
        grad=grad/(grad.square().mean().sqrt()+1e-8)
        with torch.no_grad(): x=(x+(flow(x,t,target)-guidance*grad)/ode_steps).clamp(min=0)
        if (step+1)%5==0: print(f'ODE {step+1}/{ode_steps}: frame L1 {loss.item():.6f}')
    return x.reshape_as(initial).detach()

In [ ]:
# Run flow refinement and save a checkpoint compatible with eval_env.py.
device = pick_device(); scene = Scene(device); model = load_srm(device, name=SRM_RUN)
refined = refine(model, scene)
FLOW_RUN = 'srm_colab_flow'; out = RUNS/FLOW_RUN; out.mkdir(parents=True, exist_ok=True)
state = torch.load(RUNS/SRM_RUN/'last.pt', map_location=device); state['model']['srm'] = refined.cpu()
torch.save(state, out/'last.pt')
args = json.loads((RUNS/SRM_RUN/'args.json').read_text()); args['flow_refined_from'] = SRM_RUN
(out/'args.json').write_text(json.dumps(args, indent=2))
sys.argv = ['eval_env.py', '--run', FLOW_RUN]
eval_env.main()

In [ ]:
print('Initial SRM')
display(Image(f'/content/results/{SRM_RUN}/pano.png'))
print('Flow-refined SRM')
display(Image(f'/content/results/{FLOW_RUN}/pano.png'))
# Package checkpoints and images for one-click download.
!zip -qr /content/srm_flow_results.zip /content/results /content/runs
from google.colab import files
files.download('/content/srm_flow_results.zip')

## Dataset and runtime notes

This notebook trains on the authors' real Cans capture (2,265 frames), not simulated images. It creates only `/content/cans_data`, `/content/cache`, `/content/runs`, and `/content/results`; there is no project code folder. Colab storage is temporary, so the last cell downloads the checkpoints and results. There is no ground-truth environment panorama for this real sequence; the released capture supports qualitative reconstruction and coverage measurement.